# GPU training cost

Estimate the cost of a training run. Inputs: model size, dataset size, GPU class.
Uses the standard FLOPs-per-token approximation `6 * N * D` for transformer
pre-training (Kaplan et al., OpenAI, 2020).

Reference prices are baked in (current as of 2026-05-18) and **illustrative**.
For a real estimate, check the current public rate of your cloud provider.

In [ ]:
# illustrative on-demand prices in USD per GPU-hour, ~2026
PRICES = {
    'H100_80GB': 3.00,
    'H200':      4.00,
    'A100_80GB': 1.80,
    'L4':        0.55,
    'TPU_v5p':   3.50,
}
# peak FP16/BF16 TFLOP/s per device
PEAK_TFLOPS = {
    'H100_80GB': 989,
    'H200':      989,
    'A100_80GB': 312,
    'L4':        121,
    'TPU_v5p':   459,
}

def training_cost(params_B, tokens_B, gpu='H100_80GB',
                  mfu=0.45, num_gpus=64, spot_discount=0.0):
    N = params_B * 1e9
    D = tokens_B * 1e9
    flops = 6 * N * D
    peak = PEAK_TFLOPS[gpu] * 1e12
    gpu_seconds = flops / (peak * mfu)
    wall_hours = gpu_seconds / 3600 / num_gpus
    price = PRICES[gpu] * (1 - spot_discount)
    cost = wall_hours * num_gpus * price
    return {
        'flops':         flops,
        'gpu_hours':     gpu_seconds / 3600,
        'wall_hours':    wall_hours,
        'usd':           cost,
    }

# 7B model, Chinchilla-optimal ~140B tokens, on 64xH100 at 45% MFU
for params, tokens in [(7, 140), (13, 260), (70, 1400)]:
    r = training_cost(params, tokens, gpu='H100_80GB', num_gpus=64)
    print(f'{params:3d}B params x {tokens:5d}B tokens : '
          f'{r["wall_hours"]:6.0f} wall hours, '
          f'{r["gpu_hours"]:8.0f} GPU-h, '
          f'${r["usd"]:>10,.0f}')

## Knobs

- **MFU (Model FLOPs Utilization).** Real-world MFU for large-model pretraining lands in the 30-55% band on H100 (NVIDIA / Meta / MLPerf reports, 2023-2025). Going higher requires careful kernel work and good interconnect.
- **Spot / preemptible.** 50-80% discount, but you must checkpoint and resume. See `03-training-infra/`.
- **Cluster size.** Halving the cluster doubles the wall time at the same total GPU-hours. Pick by deadline, not by cost — they're equivalent on the cost axis but not on the iteration-speed axis.
- **Precision.** FP8 (Hopper) can ~2x the effective throughput vs BF16 if your training is stable in FP8.